In [3]:
import pandas as pd
df=pd.read_csv("100_Unique_QA_Dataset.csv")
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [13]:
#Tokenization
def tokenize(text):
    text=text.lower()
    text=text.replace("?"," ")
    text=text.replace("'"," ")
    return text.split()

In [15]:
tokenize('What is the capital of France?')

['what', 'is', 'the', 'capital', 'of', 'france']

In [17]:
#vocab
vocab={'<UNK>':0}

In [39]:
def buildvocab(row):
    tokenized_question=tokenize(row['question'])
    tokenized_answer=tokenize(row['answer'])
    merged_token=tokenized_question+tokenized_answer
    for token in merged_token:
        if token not in vocab:
            vocab[token]=len(vocab)

In [41]:
df.apply(buildvocab,axis=1)

0     None
1     None
2     None
3     None
4     None
      ... 
85    None
86    None
87    None
88    None
89    None
Length: 90, dtype: object

In [45]:
len(vocab)

324

In [53]:
#convert words to numerical indices
def text_to_indices(text,vocab):
    indexed_text=[]
    for token in tokenize(text):
        if token in vocab:
            indexed_text.append(vocab[token])
        else:
            indexed_text.append(vocab['<UNK>'])
    return indexed_text

In [55]:
text_to_indices('what is your name',vocab)

[1, 2, 0, 0]

In [57]:
vocab

{'<UNK>': 0,
 'what': 1,
 'is': 2,
 'the': 3,
 'capital': 4,
 'of': 5,
 'france': 6,
 'paris': 7,
 'germany': 8,
 'berlin': 9,
 'who': 10,
 'wrote': 11,
 'to': 12,
 'kill': 13,
 'a': 14,
 'mockingbird': 15,
 'harper-lee': 16,
 'largest': 17,
 'planet': 18,
 'in': 19,
 'our': 20,
 'solar': 21,
 'system': 22,
 'jupiter': 23,
 'boiling': 24,
 'point': 25,
 'water': 26,
 'celsius': 27,
 '100': 28,
 'painted': 29,
 'mona': 30,
 'lisa': 31,
 'leonardo-da-vinci': 32,
 'square': 33,
 'root': 34,
 '64': 35,
 '8': 36,
 'chemical': 37,
 'symbol': 38,
 'for': 39,
 'gold': 40,
 'au': 41,
 'which': 42,
 'year': 43,
 'did': 44,
 'world': 45,
 'war': 46,
 'ii': 47,
 'end': 48,
 '1945': 49,
 'longest': 50,
 'river': 51,
 'nile': 52,
 'japan': 53,
 'tokyo': 54,
 'developed': 55,
 'theory': 56,
 'relativity': 57,
 'albert-einstein': 58,
 'freezing': 59,
 'fahrenheit': 60,
 '32': 61,
 'known': 62,
 'as': 63,
 'red': 64,
 'mars': 65,
 'author': 66,
 '1984': 67,
 'george-orwell': 68,
 'currency': 69,
 'unit

In [63]:
import torch
from torch.utils.data import Dataset,DataLoader

In [91]:
class QAdataset:
    def __init__(self,df,vocab):
        self.df=df
        self.vocab=vocab
    def __len__(self):
        return self.df.shape[0]
    def __getitem__(self,index):
        numerical_question=text_to_indices(self.df.iloc[index]['question'],self.vocab)
        numerical_answer=text_to_indices(self.df.iloc[index]['answer'],self.vocab)
        return torch.tensor(numerical_question),torch.tensor(numerical_answer)
    

In [93]:
dataset=QAdataset(df,vocab)

In [97]:
dataset[0]

(tensor([1, 2, 3, 4, 5, 6]), tensor([7]))

In [99]:
dataloader=DataLoader(dataset,batch_size=1,shuffle=True)
for question ,answer in dataloader:
    print(question,answer)

tensor([[  1,   2,   3, 180, 181, 182, 183]]) tensor([[184]])
tensor([[10, 75, 76]]) tensor([[77]])
tensor([[ 78,  79, 195,  81,  19,   3, 196, 197, 198]]) tensor([[199]])
tensor([[10, 29,  3, 30, 31]]) tensor([[32]])
tensor([[  1,   2,   3, 141, 117,  83,   3, 277, 278]]) tensor([[121]])
tensor([[ 78,  79, 261, 151,  14, 262, 153]]) tensor([[36]])
tensor([[ 42, 312,   2, 313,  62,  63,   3, 314, 315]]) tensor([[316]])
tensor([[42, 43, 44, 45, 46, 47, 48]]) tensor([[49]])
tensor([[ 42, 137,   2,  62,  39,   3, 322, 323]]) tensor([[6]])
tensor([[78, 79, 80, 81, 82, 83, 84]]) tensor([[85]])
tensor([[  1,   2,   3, 122, 123,  19,   3,  45]]) tensor([[124]])
tensor([[ 1,  2,  3, 92, 93, 94]]) tensor([[95]])
tensor([[10, 55,  3, 56,  5, 57]]) tensor([[58]])
tensor([[ 1,  2,  3, 24, 25,  5, 26, 19, 27]]) tensor([[28]])
tensor([[  1,   2,   3,   4,   5, 109]]) tensor([[317]])
tensor([[ 42, 255,   2, 256,  83, 257, 258]]) tensor([[259]])
tensor([[ 1,  2,  3, 50, 51, 19,  3, 45]]) tensor([[52]]

In [133]:
import torch.nn as nn
class simpleRNN(nn.Module):
    def __init__(self,vocab_size):
        super().__init__()
        self.embedding=nn.Embedding(vocab_size,embedding_dim=50)
        self.rnn=nn.RNN(50,64,batch_first=True)
        self.fc=nn.Linear(64,vocab_size)
    def forward(self,question):
        embedded_question=self.embedding(question)
        hidden,final=self.rnn(embedded_question)
        output=self.fc(final.squeeze(0))
        return output

In [135]:
x=nn.Embedding(324,embedding_dim=50)
y=nn.RNN(50,64,batch_first=True)
z=nn.Linear(64,324)
a=dataset[0][0].reshape(1,6)
print("shape of a:",a.shape)
b=x(a)
print("shape of b:",b.shape)
c,d=y(b)
print("shape of c:",c.shape)
print("shape of d:",d.shape)
e=z(d.squeeze(0))
print("shape of e:",e.shape)

shape of a: torch.Size([1, 6])
shape of b: torch.Size([1, 6, 50])
shape of c: torch.Size([1, 6, 64])
shape of d: torch.Size([1, 1, 64])
shape of e: torch.Size([1, 324])


In [137]:
learning_rate=0.001
epochs=20

In [139]:
model=simpleRNN(len(vocab))


In [141]:
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=learning_rate)

In [143]:
for epoch in range(epochs):
    total_loss=0
    for question,answer in dataloader:
        optimizer.zero_grad()
        output=model(question)
        loss=criterion(output,answer[0])
        #gradient
        loss.backward()
        #update
        optimizer.step()
        total_loss=total_loss+loss.item()
    print(f"Epoch:{epoch+1},Loss:{total_loss:4f}")

Epoch1,Loss523.157569
Epoch2,Loss456.571974
Epoch3,Loss379.228527
Epoch4,Loss316.112740
Epoch5,Loss264.220697
Epoch6,Loss216.027646
Epoch7,Loss171.951006
Epoch8,Loss134.784815
Epoch9,Loss103.225064
Epoch10,Loss79.330681
Epoch11,Loss61.485645
Epoch12,Loss47.664715
Epoch13,Loss38.172172
Epoch14,Loss30.841821
Epoch15,Loss25.743417
Epoch16,Loss21.372826
Epoch17,Loss17.946678
Epoch18,Loss15.010669
Epoch19,Loss13.076161
Epoch20,Loss11.349502


In [221]:
def predict(model,question,threshold=0.5):
    numerical_question=text_to_indices(question,vocab)
    question_tensor=torch.tensor(numerical_question).unsqueeze(0)
    output=model(question_tensor)
    #conver logits to probs
    probs=torch.nn.functional.softmax(output,dim=1)
    #find max probabilites
    value,index=torch.max(probs,dim=1)
    print(value,index)
    if value < threshold:
        print("I Don't Know")
    print(list(vocab.keys())[index])

In [223]:
predict(model,'what is the largest planet in our solar system?')

tensor([0.8998], grad_fn=<MaxBackward0>) tensor([23])
jupiter


In [225]:
list(vocab.keys())[23]

'jupiter'